# ResNet18 세 조건 비교 — 면적 10% 이하 완전 제외

## Goal
v1=기존 통과 원본, v1_5=같은 사진에 팀원 B 베드지도+승인한 직선 보정 마스크,
v2=v1_5+형상 v2 재활용. 전처리 알고리즘 자체는 바꾸지 않는다.
**10% 이하 사진은 학습, 검증, 입력 캐시, 특징 추출, 예측과 성능 집계에서 모두 제외한다.**
남는 것은 기존 통과 2,652장+재활용743장=3,395장. 제외61장은 파일명·이유 목록에만 남긴다.
세 조건은 동일3,395장으로 검증한다. v1/v1_5의 재활용 평가는 실제 게이트를 우회한 진단이다.
재활용 마스크는 기존 검토한 형상 v2 저장본을 그대로 사용한다. 불량 틈을 포함해 내부 RGB는 유지,
외부는RGB(124,116,104). 통과 마스크에 질감v2나 형상v2를 추가하지 않는다.
형상v2는 내부적으로 질감 초기후보·바깥변·영상이동·공정 라벨을 함께 사용한다. 신규 실시간 검증은 아직 없다.

## Setup
Colab GPU에서 이 노트북을 연다. Drive `산공종설`에 다음2파일을 둔다.
- `underfill_originals_4000.zip` (기존 원본)
- `resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip` (이번 새 묶음)

위에서부터 실행한다. **옛 결과/체크포인트로 이어서 학습하지 않는다.**
이 버전으로 새15회(3조건×5fold)를 실행해야 10% 이하 제외 학습의 성능을 얻는다.
T4 기준 학습1~1.5시간은 기존 추정이며 실제 환경에 따라 달라진다. 지금은 로컬 학습을 실행하지 않았다.

In [ ]:
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/산공종설')
ORIGINAL_ZIP_NAME = 'underfill_originals_4000.zip'
BUNDLE_ZIP_NAME = 'resnet18_hh_straight_shape_rescue_bundle_20261005_area_gt10_v3.zip'
RUN_NAME = 'resnet18_hh_straight_shape_rescue_20261005_area_gt10_v3_run01'
IMG_SIZE = 512       # 원본 크롭 1000x1000과 모델 입력 해상도는 다름
BATCH = 16          # GPU 메모리가 부족하면 8로 바꾸고 RUN_NAME도 변경
ROUNDS = 10         # 고정 업데이트 예산: 아래 설명 참고
SEEDS = [0]         # 우선 예비 비교 1개 시드, 최종 근거에는 한계 기록
FOLDS = [0, 1, 2, 3, 4]
RESCUE_MIN_AREA = 0.1  # 이 값 초과만 사용. 10% 이하 제외, 0으로 해제 불가
RUN_TRAINING = True  # 입력 확인만 하려면 False

### 1. GPU 확인 및 Drive 연결
본 학습은 Colab GPU에서만 실행한다. 결과와 체크포인트는 Drive에 저장한다.
런타임이 초기화되면 위에서부터 다시 실행한다. 환경/설정이 같을 때 완료한 fold는 건너뛰고
미완료 fold는 마지막 저장 round 다음부터 이어간다. round 도중 끊기면 그 round만 다시 돈다.

In [ ]:
import torch, torchvision
from google.colab import drive
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
assert torch.cuda.is_available(), '런타임 유형을 GPU로 바꿔줘.'
print('GPU:', torch.cuda.get_device_name())
drive.mount('/content/drive')
OUTPUT = DRIVE_DIR / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)

### 2. ZIP을 로컬 SSD로 풀고 파일 무결성 확인
학습 중 JPG를 Drive에서 매번 읽지 않는다. 원본과 마스크를 `/content`로 복사한다.
압축을 푼 뒤 목록·해시·기존 5-fold/30프레임 buffer를 확인한다.

In [ ]:
import zipfile, hashlib, json, shutil, subprocess, sys, os
from IPython.display import display
LOCAL = Path('/content/hh_shape_area_gt10_v3')
LOCAL.mkdir(parents=True, exist_ok=True)
BUNDLE = LOCAL / 'bundle'
ORIGINALS = LOCAL / 'originals'
CACHE = LOCAL / f'cache_{IMG_SIZE}'

def extract_zip(source, target):
    assert source.is_file(), f'파일 없음: {source}'
    local_zip = LOCAL / source.name
    # 원본은 크므로 같은 세션에서 재실행 시 파일 크기가 같은 복사본을 재사용
    if source.name == BUNDLE_ZIP_NAME or not local_zip.exists() or local_zip.stat().st_size != source.stat().st_size:
        shutil.copy2(source, local_zip)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as z:
        for member in z.infolist():
            resolved = (target / member.filename).resolve()
            assert resolved.is_relative_to(target.resolve()), 'ZIP 경로 범위 오류'
        z.extractall(target)
    print('압축 해제:', source.name)

extract_zip(DRIVE_DIR / BUNDLE_ZIP_NAME, BUNDLE)
extract_zip(DRIVE_DIR / ORIGINAL_ZIP_NAME, ORIGINALS)
hashes = json.loads((BUNDLE / 'checksums.json').read_text())
for name, expected in hashes.items():
    assert hashlib.sha256((BUNDLE / name).read_bytes()).hexdigest() == expected, name
subprocess.run([sys.executable, str(BUNDLE / 'train_resnet18.py'),
                '--bundle', str(BUNDLE), '--rescue-min-area', str(RESCUE_MIN_AREA), '--verify-only'], check=True)
print('묶음 파일 해시 검증 완료:', len(hashes))

## Data & Methods
- 원본4000장, 고정1000×1000 크롭(x250:1250,y0:1000), 모델입력512×512.
- 저장 마스크의 유지 화소 수/1,000,000으로 면적을 측정. **면적>0.1**만 사용.
- 마스크를3395장만 제공하며 작은61장은 원본 파일명 목록에만 남긴다.
- 기존 시간 블록5-fold, 경계30프레임 buffer 유지. 세 조건의 검증 사진은 동일.
- v1/v1_5는 통과 사진만 학습하고, v2는 통과+재활용을 학습한다.
- v1/v1_5 재활용 점수는 강제 투입 진단이며, 보류를 정상으로 간주하지 않는다.
- ResNet18, ImageNet 전이학습, 10 round, seed0, AdamW·증강·고정 업데이트 예산은 기존과 동일.
- `eligibility_audit.csv`에15개 분할의 최솟값과 제외 사진 혼입0건을 기록한다.
- 면적이 작아 제외된 사진은 별도 분류하지 않는다. `excluded_area.csv`는 점수가 아닌 제외 기록이다.

### 한계
클래스별 출력1회, 공정표를 쓰는 오프라인 마스크, 회색 면적 지름길 미해결.
OOF는 새 출력 일반화 검증이 아니다. 이전804장 학습 결과는 이번 버전 결과가 아니다.

In [ ]:
import pandas as pd
manifest = pd.read_csv(BUNDLE / 'manifest.csv')
prep = json.loads((BUNDLE / 'preparation.json').read_text(encoding='utf-8'))
display(pd.read_csv(BUNDLE / 'coverage.csv'))
sys.path.insert(0,str(BUNDLE))
from area_policy import eligible
selected = pd.DataFrame([r for r in manifest.to_dict('records') if eligible(r,RESCUE_MIN_AREA)]).reset_index(drop=True)
candidates = manifest[manifest.original_cohort.isin(['primary','rescue'])]
excluded = candidates[candidates.mask_area <= RESCUE_MIN_AREA]
assert len(selected)+len(excluded)==3456
assert (selected.mask_area>RESCUE_MIN_AREA).all()
assert not set(selected.source)&set(excluded.source)
print('학습/검증/추론 후보:', len(selected),'완전 제외:',len(excluded))
display(selected.groupby(['cohort','label']).mask_area.agg(['count','min','median','max']))
display(excluded[['source','mask_pixels','mask_area','exclusion_reason']].head())
N_PRIMARY = int((selected.cohort == "primary").sum())
N_RESCUE = int((selected.cohort == "rescue").sum())

### 3. 실제 학습 입력 생성 및 확인
원본·마스킹 입력 두 벌을 메모리맵으로 저장한다(512일 때 약 5.1 GiB, 로컬 SSD).
영역 내부 픽셀이 유지되는지, 모든 입력 사진·마스크의 SHA256이 맞는지 확인한다.
아래 미리보기에서 **원본과 마스킹 결과를 직접 보고** 학습 셀로 넘어간다.

In [ ]:
def logged_run(command, log_path):
    env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', env=env)
        for line in process.stdout:
            log.write(line); log.flush()
            # 다운로드/긴 로그는 파일로; 주요 진행만 노트북에 출력
            if any(word in line for word in ('입력 ', '완료', 'round ', 'Traceback', 'Error', '이미 완료', '보류 ', '재활용 면적')):
                print(line.rstrip(), flush=True)
        result = process.wait()
    assert result == 0, f'실행 실패: {log_path} 확인'

common = [sys.executable, str(BUNDLE / 'train_resnet18.py'),
          '--bundle', str(BUNDLE), '--originals', str(ORIGINALS),
          '--cache', str(CACHE), '--output', str(OUTPUT),
          '--size', str(IMG_SIZE), '--batch', str(BATCH), '--rounds', str(ROUNDS),
          '--rescue-min-area', str(RESCUE_MIN_AREA)]
logged_run(common + ['--prepare-only'], OUTPUT / 'input_preparation.log')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
raw = np.load(CACHE / 'raw.npy', mmap_mode='r')
masked = np.load(CACHE / 'masked.npy', mmap_mode='r')
sample = []
for cohort in ('primary', 'rescue'):
    for label in ('normal', 'underfill_50'):
        sample.append(int(selected[(selected.cohort == cohort) & (selected.label == label)].index[0]))
for frame in (2127, 3654):
    found = selected.index[selected.frame_index == frame].tolist()
    if found: sample.append(found[0])
fig, axes = plt.subplots(len(sample), 2, figsize=(9, 3.5 * len(sample)), squeeze=False)
for row, index in enumerate(sample):
    item = selected.iloc[index]
    method = 'HH + approved straight boundary' if item.cohort == 'primary' else 'Shape v2 rescue'
    for col, (array, title) in enumerate(((raw, 'Raw fixed crop'), (masked, method + ' masked'))):
        axes[row,col].imshow(array[index].transpose(1,2,0))
        axes[row,col].axis('off')
        axes[row,col].set_title(f'{title}: {item.label} #{item.frame_index}\n{item.mask_status}, area={item.mask_area:.3%}', fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUT / 'actual_input_preview.png', dpi=120, bbox_inches='tight')
plt.show()

### 4. 학습 실행
아래 셀이 실제 GPU 학습이다. 아직 결과가 없으므로 성능이 좋아질 것이라고 단정하지 않는다.
진행 및 체크포인트는 Drive에 저장한다. 설정을 바꿀 때는 RUN_NAME도 바꾼다.
중단한 실행은 같은 설정과 같은 환경에서 재개한다. 체크포인트는 본인 실행 파일만 사용한다.

In [ ]:
if RUN_TRAINING:
    command = common + ['--seeds'] + list(map(str, SEEDS)) + ['--folds'] + list(map(str, FOLDS))
    logged_run(command, OUTPUT / 'training.log')
else:
    print('입력 확인까지만 실행. 학습하지 않았어.')

## Results / Checks
OOF는 면적 조건을 통과한 동일3,395장에 대해서만 나온다.
`all_common`과`rescue`에 작은61장이 포함되지 않는다. `rescue_small` 진단 점수도 만들지 않는다.
`pipeline_accepted`는 조건별 판정 범위가 다르다. 같은 평가 사진의 v1_5↔v2를 비교한다.
`eligibility_audit.csv`와`eligibility_policy.json`으로 대상 사진을 확인한다.

In [ ]:
if (OUTPUT / 'summary.csv').exists():
    summary = pd.read_csv(OUTPUT / 'summary.csv')
    columns = ['version','seed','cohort','completed_folds','all_folds_complete',
               'n','accuracy','balanced_accuracy','auroc','fp','fn']
    display(summary[columns].round(4))
    display(pd.read_csv(OUTPUT / 'paired_comparison.csv').round(4))
    print('[보류와 최장 연속 보류]'); display(pd.read_csv(OUTPUT / 'holds.csv').round(4))
    print('[5장 다수결]'); display(pd.read_csv(OUTPUT / 'vote.csv').round(4))
    plot_data = summary[summary.cohort.isin(['primary','rescue'])]
    if plot_data.all_folds_complete.all():
        for seed in sorted(plot_data.seed.unique()):
            table = plot_data[plot_data.seed == seed].pivot(index='version', columns='cohort', values='balanced_accuracy')
            ax = table.reindex(['v1','v1_5','v2']).plot.bar(figsize=(8,4), color=['#2463a2','#e39c24'], rot=0)
            ax.set_ylim(0,1); ax.set_ylabel('Balanced accuracy (OOF)')
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            ax.set_title(f'Same validation photos: primary n={N_PRIMARY} / rescue n={N_RESCUE}, seed {seed}')
            for index, container in enumerate(ax.containers):
                if index == 1:
                    for bar in container: bar.set_hatch('//')
                ax.bar_label(container, labels=[f'{bar.get_height():.1%}' for bar in container], padding=3, fontsize=9)
            ax.grid(axis='y', alpha=.2); ax.figure.tight_layout()
            ax.figure.savefig(OUTPUT / f'oof_balanced_accuracy_seed{seed}.png', dpi=150)
            plt.show()
    else:
        print('일부 fold만 완료됨. 최종 비교로 해석하지 말 것.')
else:
    print('완료한 학습 결과 없음. 실제 학습 후 이 셀을 다시 실행해줘.')

## Next Steps
Drive 결과를 확인한 다음 논문/팀 비교에 쓸 CSV·설정·로그를 아래 ZIP으로 내려받는다.
큰 모델 체크포인트는 ZIP에서 제외하며 Drive에 유지한다.
그래프는 실제 학습 후 생성되므로 이번 작성 단계에서는 확인하지 못했다.
Colab 전체 실행도 아직 하지 않았다. 오류가 나면 로그와 마지막 오류 셀을 공유한다.
결과 보관을 확인한 후 **런타임 → 연결 해제 및 삭제**로 GPU 런타임을 반납한다.

In [ ]:
result_zip = DRIVE_DIR / f'{RUN_NAME}_results.zip'
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT.rglob('*'):
        if path.is_file() and path.suffix in ('.csv','.json','.log','.png'):
            z.write(path, path.relative_to(OUTPUT))
print('결과 위치:', OUTPUT)
print('공유용 결과 ZIP:', result_zip)

### 런타임 반납
결과 ZIP 위치를 확인한 뒤 실행한다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()